### 引入套件

In [2]:
import argparse, gym, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.distributions import Categorical

### Actor & Critic Network

$A(s_t, a_t) = r_{t+1} + \gamma V_v(s_{t+1}) - V_v(s_t)$

The advantage function $ A(s_t, a_t) $ measures how much better or worse taking action $ a_t $ in state $ s_t $ is compared to the expected outcome. 

 - $r_{t+1} + \gamma V_v(s_{t+1})$ is represented by the actual discounted rewards 

#### The idea behind

If $ A(s_t, a_t) $ is positive, it means the action $ a_t $ was better than expected, so the policy should be be adjusted to increase the probability of selecting this action in similar situations. 

If it’s negative, the action was worse than expected, so the policy should reduce the probability of selecting this action.




In [2]:
class Actor(nn.Module): # A(s, a) 
    def __init__(self, state_size=8, num_actions=4):
        super(Actor, self).__init__()
        self.fc1 = nn.Linear(state_size, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 16)
        self.fc4 = nn.Linear(16, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.log_softmax(self.fc4(x),dim=-1)
        return x

class Critic(nn.Module): # V(s)
    def __init__(self, state_size=8):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(state_size, 16)
        self.dp1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dp1(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## A2C Alogrithm 

In [4]:
## A2C Alogrithm 

# Define Advantage Actor-Critic (A2C) agent
class A2C(object):
    def __init__(self, env, args):
        pass

    # Select action using the actor network
    def select_action(self, state):
        pass

    # Play an entire episode and record metrics
    def play_episode(self, e):
        pass

    # Optimize the actor and critic based on the recorded experience
    def optimize(self, rewards, log_probs, values):
        pass

    # Train the agent for the specified number of episodes
    def train(self, num_episodes):
        pass


### Sketch

In [5]:
def select_action(self, state):
    '''
    Return
    - action: the action taken in this state
    - log_probs[action]: the probabilities of taken this action
    - value: expected outcome of the current state
    '''
    log_probs = self.actor(state) # ex: [0.1, 0.2, 0.3, 0.4]: probabilities of each action
    value = self.critic(state) # V(s_t)
    action = ... # sample from log_probs
    return action, log_probs[action], value # return three 1x1 tensors

In [ ]:
def play_episode(self, e):
    '''
    Return
    - steps: current timestep
    - rewards: rewards of each step, ex: [r_0, r_1, ..., r_T]
    - log_probs: prob. of each taken action, ex: [log_probs_0, log_probs_1, ..., log_probs_T]
    - values: value of each state, ex: [V(s_0), V(s_1), ..., V(s_T)]
    '''
    steps = ... 
    rewards = ... 
    log_probs = ... 
    values = ...
    return steps, rewards, torch.cat(log_probs), torch.cat(values)

### 1. Discounted Return $R$

$A(s_t, a_t) = r_{t+1} + \gamma V_v(s_{t+1}) - V_v(s_t)$

For each time step $t$, the discounted return  $R[t]$ is calculated as follows:


$R[t] = \gamma^N \cdot V_{\text{end}} + \sum_{k=0}^{\min(N, T - t) - 1} \gamma^k \cdot r_{t + k} \cdot 10^{-2}$

where:
- $ \gamma $ is the discount factor,
- $ V_{\text{end}} $ is the estimated value $ N $ steps ahead ($ 0 $ if $ t + N \geq T $),
- $ r_{t + k} $ is the reward received $ k $ steps after time $ t $.


That is, 

If $t+N \leq T$, then $R[t] = r_t + \gamma r_{t+1} + \dots + \gamma^{N} r_{t+N}$

If $t+N > T$, then $R[t] = r_t + \gamma r_{t+1} + \dots + \gamma^{T-t} r_{T-t}$

In [ ]:
# Calculate discounted reward R and advantage
rewards = [1, 0, 1, 0]  # Rewards for each step
values = [1, 2, 3, 4]  # Values for each step
log_probs = [0.1, 0.2, 0.3, 0.4]  # Log probabilities for each step

T = ...  # Total steps in the episode
N = ...  # N-step lookahead for calculating returns
R = np.zeros(T, dtype=np.float32)  # Initialize array to store discounted returns for each step
gamma = 0.99  # Discount factor

for t in reversed(range(T)):
    V_end = 0 if (t+N >= T) else values[t+N].data
    R[t] = (gamma**N * V_end) + sum([gamma**k * rewards[t+k]*1e-2 for k in range(min(N, T-t))])

R = Variable(torch.Tensor(R), requires_grad=False)

### 2. Critic Loss $ \mathcal{L}_{\text{critic}} $

The critic loss is computed as the **mean squared error** between the return $ R $ and the value $ V(s_t) $:

$
\mathcal{L}_{\text{critic}} = \frac{1}{T} \sum_{t=0}^{T-1} \left( R[t] - V(s_t) \right)^2
$

where:
- $ V(s_t) $ is the critic’s estimate of the state value at time $ t $,
- $ R[t] $ is the computed return at time $ t $.

In [ ]:
loss_critic = ((R - values)**2).mean()

### 3. Actor Loss $ \mathcal{L}_{\text{actor}} $

The actor loss is based on the ```advantage``` (difference between the return $ R $ and the estimated value $ V $) weighted by the ```negative log probability``` of the chosen actions:

$
\mathcal{L}_{\text{actor}} = -\frac{1}{T} \sum_{t=0}^{T-1} \left( R[t] - V(s_t) \right) \cdot \log \pi(a_t | s_t)
$

where:
- $ \pi(a_t | s_t) $ is the probability of action $ a_t $ given state $ s_t $,
- $ \log \pi(a_t | s_t) $ is the log probability of the action chosen by the policy,
- $ R[t] - V(s_t) $ represents the advantage at time $ t $.

In [ ]:
loss_actor = ((R - values.detach()) * -log_probs).mean()

## The A2C agent

In [ ]:
# Define Advantage Actor-Critic (A2C) agent
class A2C(object):
    def __init__(self, env, args):
        super(A2C, self).__init__()
        # Initialize environment and A2C model components
        self.env = env
        self.actor = Actor()
        self.critic = Critic()
        
        # Set up optimizers for actor and critic
        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=args.lr_actor)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=args.lr_critic)
        
        # Hyperparameters and settings from args
        self.N_steps, self.gamma = args.N_steps, args.gamma
        self.num_episodes, self.test_episodes = args.num_episodes, args.test_episodes
        self.expt_name, self.save_path = args.expt_name, args.save_path
        self.test_freq, self.save_freq = args.test_freq, args.save_freq
        
        # Containers for training metrics
        self.train_rewards, self.test_rewards = [], []
        self.train_steps, self.test_steps = [], []
        self.losses_actor, self.losses_critic = [], []

    # Select action using the actor network
    def select_action(self, state):
        state = Variable(torch.Tensor(state))
        log_probs = self.actor(state)
        value = self.critic(state)
        action = Categorical(log_probs.exp()).sample()
        return action.data.cpu().numpy()[0], log_probs[action], value

    # Play an entire episode and record metrics
    def play_episode(self, e):
        state = self.env.reset()
        steps = 0
        rewards = []
        log_probs = []
        values = []

        # Continue until the episode is complete
        while True:
            action, log_prob, value = self.select_action(state)
            state, reward, is_terminal, _ = self.env.step(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            values.append(value)
            steps +=1
            if is_terminal:
                break

        return steps, rewards, torch.cat(log_probs), torch.cat(values)

    # Optimize the actor and critic based on the recorded experience
    def optimize(self, rewards, log_probs, values):
        T = len(rewards)  # Total steps in the episode
        N = self.N_steps  # N-step lookahead for calculating returns
        R = np.zeros(T, dtype=np.float32)  # Initialize array to store discounted returns for each step
        loss_actor = 0
        loss_critic = 0

        # Calculate discounted reward R and advantage
        # Section 1. Discounted Return 
        for t in reversed(range(T)):
            V_end = 0 if (t+N >= T) else values[t+N].data
            R[t] = (self.gamma**N * V_end) + sum([self.gamma**k * rewards[t+k]*1e-2 for k in range(min(N, T-t))])
        R = Variable(torch.Tensor(R), requires_grad=False)
        
        # Calculate actor and critic losses
        # Section 2. Critic Loss
        loss_critic = ((R - values)**2).mean()
        # Section 3. Actor Loss
        loss_actor = ((R - values.detach()) * -log_probs).mean()

        # Perform backpropagation
        self.optimizer_actor.zero_grad()
        self.optimizer_critic.zero_grad()
        loss_actor.backward()
        loss_critic.backward()
        self.optimizer_actor.step()
        self.optimizer_critic.step()

        # Save losses for tracking
        self.losses_actor.append(loss_actor.data.cpu().numpy()[0])
        self.losses_critic.append(loss_critic.data.cpu().numpy()[0])

    # Train the agent for the specified number of episodes
    def train(self, num_episodes):
        for e in range(num_episodes):
            steps, rewards, log_probs, values = self.play_episode(e)
            self.train_rewards.append(sum(rewards))
            self.train_steps.append(steps)
            self.optimize(rewards, log_probs, values)
